# Demo 3 · StructArray 父子混合检索（一查询三答卷）

复现官方 `structarray-search`：**30 个行车视频**是父记录，每个视频的 18 条**目标观察**（3 个时间相位 × 6 个目标）是子记录，全部装进一个 `ARRAY<STRUCT>` 字段，一张表搞定，不建两张表。

同一个查询，三种答法：**父向量**（找整段视频）、**子向量 + 按视频聚合**（找画面里具体目标）、**双路加权融合**。

嵌入用你自己的 API（推荐百炼 `text-embedding-v4`，1024 维）：570 条文本，1~2 分钟跑完，不需要下载模型。

In [1]:
%pip install -q pymilvus==3.0.2 requests numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: C:\Users\liuxiao\GenericAgent\temp\milvus3_lab\venv\Scripts\python.exe -m pip install --upgrade pip


## 1. 下载场景数据，填你的嵌入 API

In [2]:
import json, os, requests, numpy as np

SCENES_URL = ("https://raw.githubusercontent.com/zc277584121/milvus3-demos/main/"
              "demos/structarray-search/data/synthetic-driving-scenes-r1/scenes.json")
data = requests.get(SCENES_URL, timeout=60).json()
scenes = data["scenes"]
phases = [(p["frame_offset"], p["motion"]) for p in data["observation_phases"]]
print("视频:", len(scenes), "| 相位:", phases)

# ====== 填你的嵌入 API（百炼兼容端点） ======
EMB_BASE = "https://dashscope.aliyuncs.com/compatible-mode/v1"
EMB_MODEL = "text-embedding-v4"          # 1024 维
EMB_KEY = os.environ.get("EMB_KEY", "")  # 或直接把 key 串贴进引号
# ==========================================

assert EMB_KEY, "请先填写 EMB_KEY（阿里云百炼的 API Key）"

def embed(texts, batch=10):
    vecs = []
    for i in range(0, len(texts), batch):
        r = requests.post(EMB_BASE + "/embeddings",
                          headers={"Authorization": "Bearer " + EMB_KEY},
                          json={"model": EMB_MODEL, "input": texts[i:i + batch],
                                "dimensions": 1024},
                          timeout=60)
        r.raise_for_status()
        vecs += [d["embedding"] for d in r.json()["data"]]
    return np.asarray(vecs, dtype=np.float32)
print("embed() ready")

视频: 30 | 相位: [(0, 'approaches steadily'), (12, 'continues through the camera view'), (24, 'moves farther ahead')]
embed() ready


## 2. 构造 540 条观察文本，一次性生成向量

In [3]:
summaries, obs_rows = [], []
for i, sc in enumerate(scenes):
    summaries.append(sc["summary"])
    for pi, (off, motion) in enumerate(phases):
        for oi, o in enumerate(sc["objects"]):
            otype, color, vtype, orient, lights, bbox = o
            obs_rows.append({
                "video_id": sc["video_id"],
                "description": f"A {color} {vtype} {motion} {orient}; the {otype} has its lights {lights}.",
                "object_type": otype,
                "frame_id": (i + 1) * 1000 + off,
                "bbox": [int(x) for x in bbox],
            })
print("观察条数:", len(obs_rows))
desc_vecs = embed([r["description"] for r in obs_rows])   # 540 x 1024
sum_vecs = embed(summaries)                               # 30 x 1024
print("向量:", desc_vecs.shape, sum_vecs.shape)

观察条数: 540


向量: (540, 1024) (30, 1024)


## 3. 建集合：一个 ARRAY<STRUCT> 装下全部子记录

`summary_vector` 走父级检索，`observations[description_vector]` 走子级检索，两条索引各管一路。

In [4]:
from pymilvus import MilvusClient, DataType, AnnSearchRequest, WeightedRanker

client = MilvusClient(uri="http://127.0.0.1:19530")
COLL = "demo3_driving"
if client.has_collection(COLL):
    client.drop_collection(COLL)

schema = client.create_schema(auto_id=False, enable_dynamic_field=False)
schema.add_field("video_id", DataType.VARCHAR, is_primary=True, max_length=64)
schema.add_field("video_summary", DataType.VARCHAR, max_length=1024)
schema.add_field("summary_vector", DataType.FLOAT_VECTOR, dim=1024)
obs = client.create_struct_field_schema()
obs.add_field("description", DataType.VARCHAR, max_length=512)
obs.add_field("description_vector", DataType.FLOAT_VECTOR, dim=1024)
obs.add_field("object_type", DataType.VARCHAR, max_length=32)
obs.add_field("frame_id", DataType.INT64)
for k in ("x1", "y1", "x2", "y2"):
    obs.add_field(f"bbox_{k}", DataType.INT64)
schema.add_field("observations", DataType.ARRAY, element_type=DataType.STRUCT,
                 struct_schema=obs, max_capacity=32)

ip = client.prepare_index_params()
ip.add_index(field_name="summary_vector", index_name="idx_summary", index_type="HNSW",
             metric_type="COSINE", params={"M": 16, "efConstruction": 64})
ip.add_index(field_name="observations[description_vector]", index_name="idx_obs",
             index_type="HNSW", metric_type="COSINE", params={"M": 16, "efConstruction": 64})
client.create_collection(COLL, schema=schema, index_params=ip)

by_video = {}
for i, r in enumerate(obs_rows):
    by_video.setdefault(r["video_id"], []).append(i)

rows = []
for si, sc in enumerate(scenes):
    rows.append({
        "video_id": sc["video_id"],
        "video_summary": sc["summary"],
        "summary_vector": sum_vecs[si].tolist(),
        "observations": [{
            "description": obs_rows[j]["description"],
            "description_vector": desc_vecs[j].tolist(),
            "object_type": obs_rows[j]["object_type"],
            "frame_id": obs_rows[j]["frame_id"],
            "bbox_x1": obs_rows[j]["bbox"][0], "bbox_y1": obs_rows[j]["bbox"][1],
            "bbox_x2": obs_rows[j]["bbox"][2], "bbox_y2": obs_rows[j]["bbox"][3],
        } for j in by_video[sc["video_id"]]],
    })
client.insert(COLL, data=rows)
client.flush(COLL)
client.load_collection(COLL)
print(client.get_collection_stats(COLL))

{'row_count': 30}


## 4. 一个查询，三路答案

In [5]:
qvec = embed(["A bicycle moving through the intersection with its lights on."])[0]

# A 父向量：整段视频
ra = client.search(COLL, data=[qvec], anns_field="summary_vector",
                   search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                   limit=3, output_fields=["video_id", "video_summary"])
print("== A 父向量 ==")
for h in ra[0]:
    print(f'{h["distance"]:.4f}  {h["video_id"]}')

# B 子向量 + 按视频归并（每段视频只出 1 条，附最优子元素）
rb = client.search(COLL, data=[qvec], anns_field="observations[description_vector]",
                   search_params={"metric_type": "COSINE", "params": {"ef": 64}},
                   group_by_field="video_id",
                   limit=5, output_fields=["video_id", "video_summary"])
print("== B 子向量(按视频聚合) ==")
for h in rb[0]:
    print(f'{h["distance"]:.4f}  {h["video_id"]}')

# C 双路融合
parent_req = AnnSearchRequest(data=[qvec], anns_field="summary_vector",
                              param={"metric_type": "COSINE", "params": {"ef": 64}}, limit=10)
child_req = AnnSearchRequest(data=[qvec], anns_field="observations[description_vector]",
                             param={"metric_type": "COSINE",
                                    "params": {"ef": 64, "element_scope": {
                                        "collapse": {"strategy": "topk_sum", "topk": 3}}}},
                             limit=40)
rc = client.hybrid_search(COLL, reqs=[parent_req, child_req],
                          ranker=WeightedRanker(0.4, 0.6), limit=3,
                          output_fields=["video_id", "video_summary"])
print("== C 加权融合(0.4 父 / 0.6 子) ==")
for h in rc[0]:
    print(f'{h["distance"]:.4f}  {h["video_id"]}')

== A 父向量 ==
0.6981  synthetic-drive-001
0.6872  synthetic-drive-005
0.6445  synthetic-drive-002
== B 子向量(按视频聚合) ==
0.7390  synthetic-drive-005
0.7315  synthetic-drive-001
0.7315  synthetic-drive-017
0.7298  synthetic-drive-016
0.7297  synthetic-drive-024
== C 加权融合(0.4 父 / 0.6 子) ==
1.2976  synthetic-drive-005
1.2928  synthetic-drive-001
1.2678  synthetic-drive-027


## 5. 看点与坑

1. **一表两用**：父子同表，`observations` 数组最多 32 个元素，每个元素自带 1024 维向量；
2. **两种归并**：B 路用 `group_by_field="video_id"`（每组返回最优子元素，hit 自带 offset）；C 路融合时用 `element_scope.collapse`（策略 topk_sum 取子分前 3 求和，还有 max / sum / avg / topk_avg）；
3. **两路分数量纲**：父路是摘要相似度、子路是聚合并相似度，WeightedRanker 归一化后融合。

## 清理（可选）

In [6]:
client.drop_collection("demo3_driving")
print("done")

done


跑完这条链，你能亲手验证：

- 同一查询三种粒度：A 找"整段像的视频"，B 找"画面里有这个目标的视频"，C 两者加权；
- 540 条子观察 + 30 条父摘要全部在一张表里，没有再建关系表；
- 嵌入只花几分钟 API 调用，不需要 GPU，也不需要下载模型。